In [1]:
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf

from visgen.datasets import IRAVEN
from visgen.models import get_model
from visgen.models.resnet_mixer import RepresentationMixer

In [2]:
def num_pos(constellation_code: str) -> int:
    if constellation_code == "all":
        return max(pos[-1] for pos in IRAVEN.POSITION_MAP.values())
    constellation_name = IRAVEN.CONSTELLATION_CODE_TO_NAME[constellation_code]
    return len(IRAVEN.POSITION_MAP[constellation_name])


# Necesario para datasets IRAVEN con ${num_pos:...}
OmegaConf.register_new_resolver("num_pos", num_pos, replace=True)

In [25]:
BASE_CFG = Path("configs/base.yml")
EXPERIMENT_CFG = Path("configs/experiments/iid.yml")
DATASET_CFGS = sorted(Path("configs/datasets").glob("*.yml"))

MODEL_CFGS = {
    "resnet18": Path("configs/models/resnet18.yml"),
    #"resnet18_mixer": Path("configs/models/resnet18_mixer.yml"),
    "resnet18_mixer_t64": Path("configs/models/resnet18_mixer_rp64_all_cases.yml"),
    "resnet18_mixer_t128": Path("configs/models/resnet18_mixer_rp128_all_cases.yml"),
    "resnet18_mixer_t256": Path("configs/models/resnet18_mixer_rp256_all_cases.yml"),
    "resnet18_mixer_alg": Path("configs/models/resnet18_algebraic_non_iid.yml"),
    "split_resnet": Path("configs/models/split.yml"),
    #"split_resnet_mixer": Path("configs/models/split_resnet_mixer.yml"),
    "split_resnet_mixer_t64": Path("configs/models/split_resnet_mixer_red_64.yml"),
    "split_resnet_mixer_t128": Path("configs/models/split_resnet_mixer_red_128.yml"),
    "split_resnet_mixer_t256": Path("configs/models/split_resnet_mixer_red_256.yml"),
    "split_resnet_mixer_alg": Path("configs/models/split_resnet_algebraic_non_iid.yml"),
    #"split_resnet_mixer_reduced_rep": Path("configs/models/split_resnet_mixer_reduced_rep.yml"),
    "ed": Path("configs/models/ed.yml"),
    "ed_mixer": Path("configs/models/ed_mixer.yml"),
    "ed_mixer_t64": Path("configs/models/ed_mixer_rp64.yml"),
    "ed_mixer_t128": Path("configs/models/ed_mixer_rp128.yml"),
    "ed_mixer_t256": Path("configs/models/ed_mixer_rp256.yml"),
    "ed_mixer_alg": Path("configs/models/ed_algebraic_non_iid.yml"),
}

MIXER_REP_PIECE_DIMS = (128, 256)

In [26]:

def build_cfg(dataset_cfg_path: Path, model_cfg_path: Path):
    cfg = OmegaConf.merge(
        OmegaConf.load(BASE_CFG),
        OmegaConf.load(EXPERIMENT_CFG),
        OmegaConf.load(dataset_cfg_path),
        OmegaConf.load(model_cfg_path),
    )

    cfg.device = "cpu"

    if not cfg.data.training.targets:
        all_targets = "_".join(att.name for att in cfg.data.training.attributes)
        cfg.data.training.targets = all_targets
        cfg.data.testing.targets = all_targets

    # --- defaults para ED ---

    if str(cfg.model.arch).startswith("ed"):
         if "preprocessing" not in cfg.model or cfg.model.preprocessing is None:
             cfg.model.preprocessing = []
         if "path" not in cfg.model or cfg.model.path is None:
            cfg.model.path = f"out/{cfg.model.arch}_{cfg.data.training.dataset}"
    #if cfg.model.arch == "ed":
    #    if "preprocessing" not in cfg.model or cfg.model.preprocessing is None:
    #        cfg.model.preprocessing = []
    #    if "path" not in cfg.model or cfg.model.path is None:
            # ruta por dataset para evitar colisiones
            #cfg.model.path = f"out/ed_{cfg.data.training.dataset}"
      

    return cfg

def count_parameters(module, trainable_only=False):
    params = module.parameters()
    if trainable_only:
        params = [p for p in params if p.requires_grad]
    return sum(p.numel() for p in params)


def summarize_model_dataset(model_name: str, dataset_cfg_path: Path, rep_piece_dim: int | None = None):
    cfg = build_cfg(dataset_cfg_path, MODEL_CFGS[model_name])

    if rep_piece_dim is not None:
        cfg.model.mixer.rep_piece_dim = rep_piece_dim

    model = get_model(cfg)

    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)

    mixer_params = 0
    mixer_trainable_params = 0
    if hasattr(model, "mixer") and model.mixer is not None:
        mixer_params = count_parameters(model.mixer)
        mixer_trainable_params = count_parameters(model.mixer, trainable_only=True)

    model_label = model_name
    if rep_piece_dim is not None:
        model_label = f"{model_name}_rep{rep_piece_dim}"

    return {
        "dataset_cfg": dataset_cfg_path.stem,
        "dataset": cfg.data.training.dataset,
        "modelo": model_label,
        "params_totales": total_params,
        "params_entrenables": trainable_params,
        "params_mixer": mixer_params,
        "params_mixer_entrenables": mixer_trainable_params,
        "params_training": total_params,
        "params_inferencia_sin_mixer": total_params - mixer_params,
        "mixer_rep_piece_dim": rep_piece_dim,
    }


In [27]:
# Tabla principal: modelo - dataset
rows = []

for dataset_cfg in DATASET_CFGS:
    for model_name in MODEL_CFGS:
        if model_name == "split_resnet_mixer_reduced_rep":
            for rep_piece_dim in MIXER_REP_PIECE_DIMS:
                rows.append(
                    summarize_model_dataset(
                        model_name,
                        dataset_cfg,
                        rep_piece_dim=rep_piece_dim,
                    )
                )
        else:
            rows.append(summarize_model_dataset(model_name, dataset_cfg))

model_dataset_df = pd.DataFrame(rows).sort_values(
    ["dataset", "modelo", "dataset_cfg"]
).reset_index(drop=True)
model_dataset_df


,dataset_cfg,dataset,modelo,params_totales,params_entrenables,params_mixer,params_mixer_entrenables,params_training,params_inferencia_sin_mixer,mixer_rep_piece_dim
0,cars3d,cars3d,ed,33531072,33529536,0,0,33531072,33531072,None
1,cars3d_iid,cars3d,ed,33531072,33529536,0,0,33531072,33531072,None
2,cars3d_non_iid,cars3d,ed,33531072,33529536,0,0,33531072,33531072,None
3,cars3d,cars3d,ed_mixer,65027776,65026240,31496704,31496704,65027776,33531072,None
4,cars3d_iid,cars3d,ed_mixer,65027776,65026240,31496704,31496704,65027776,33531072,None
...,...,...,...,...,...,...,...,...,...,...
283,shapes3d_iid,shapes3d,split_resnet_mixer_t256,48984171,48984171,31496704,31496704,48984171,17487467,None
284,shapes3d_non_iid,shapes3d,split_resnet_mixer_t256,48984171,48984171,31496704,31496704,48984171,17487467,None
285,shapes3d,shapes3d,split_resnet_mixer_t64,17695851,17695851,4338304,4338304,17695851,13357547,None
286,shapes3d_iid,shapes3d,split_resnet_mixer_t64,17695851,17695851,4338304,4338304,17695851,13357547,None


In [28]:
# Tabla training vs inferencia (sin mixer)
training_vs_inference_df = model_dataset_df[
    [
        "dataset_cfg",
        "dataset",
        "modelo",
        "params_training",
        "params_inferencia_sin_mixer",
        "params_mixer",
    ]
].copy()

training_vs_inference_df["diferencia_training_vs_inferencia"] = (
    training_vs_inference_df["params_training"]
    - training_vs_inference_df["params_inferencia_sin_mixer"]
)

training_vs_inference_df

,dataset_cfg,dataset,modelo,params_training,params_inferencia_sin_mixer,params_mixer,diferencia_training_vs_inferencia
0,cars3d,cars3d,ed,33531072,33531072,0,0
1,cars3d_iid,cars3d,ed,33531072,33531072,0,0
2,cars3d_non_iid,cars3d,ed,33531072,33531072,0,0
3,cars3d,cars3d,ed_mixer,65027776,33531072,31496704,31496704
4,cars3d_iid,cars3d,ed_mixer,65027776,33531072,31496704,31496704
...,...,...,...,...,...,...,...
283,shapes3d_iid,shapes3d,split_resnet_mixer_t256,48984171,17487467,31496704,31496704
284,shapes3d_non_iid,shapes3d,split_resnet_mixer_t256,48984171,17487467,31496704,31496704
285,shapes3d,shapes3d,split_resnet_mixer_t64,17695851,13357547,4338304,4338304
286,shapes3d_iid,shapes3d,split_resnet_mixer_t64,17695851,13357547,4338304,4338304


In [32]:
# Mixer standalone usando configuraciones transformer 64/128/256
mixer_standalone_variants = [
    ("representation_mixer_standalone_t64", "resnet18_mixer_t64"),
    ("representation_mixer_standalone_t128", "resnet18_mixer_t128"),
    ("representation_mixer_standalone_t256", "resnet18_mixer_t256"),
]

mixer_rows = []
for label, cfg_name in mixer_standalone_variants:
    cfg_mixer = build_cfg(DATASET_CFGS[0], MODEL_CFGS[cfg_name])
    standalone_mixer = RepresentationMixer(
        emb_dim=cfg_mixer.model.emb_dim,
        num_layers=cfg_mixer.model.mixer.num_layers,
        num_heads=cfg_mixer.model.mixer.num_heads,
        dropout=cfg_mixer.model.mixer.dropout,
    )

    standalone_mixer_params = count_parameters(standalone_mixer)
    standalone_mixer_trainable = count_parameters(standalone_mixer, trainable_only=True)

    mixer_rows.append(
        {
            "modelo": label,
            "params_totales": standalone_mixer_params,
            "params_entrenables": standalone_mixer_trainable,
            "params_mixer": standalone_mixer_params,
            "params_training": standalone_mixer_params,
            "params_inferencia_sin_mixer": 0,
        }
    )

mixer_standalone_df = pd.DataFrame(mixer_rows)

In [33]:
# Tabla final combinada
final_df = pd.concat(
    [
        model_dataset_df,
        mixer_standalone_df.assign(dataset_cfg="-", dataset="-"),
    ],
    ignore_index=True,
)

filter_df = final_df['dataset_cfg'].isin(['cars3d','dsprites','iraven','mpi3d','shapes3d','clevr'])
final_df = final_df[filter_df]

final_df = (
    final_df.groupby(["dataset", "modelo"], dropna=False)
      .mean(numeric_only=True)
      .reset_index()
)


final_df

,dataset,modelo,params_totales,params_entrenables,params_mixer,params_mixer_entrenables,params_training,params_inferencia_sin_mixer
0,cars3d,ed,33531072.0,33529536.0,0.0,0.0,33531072.0,33531072.0
1,cars3d,ed_mixer,65027776.0,65026240.0,31496704.0,31496704.0,65027776.0,33531072.0
2,cars3d,ed_mixer_alg,33531072.0,33529536.0,0.0,0.0,33531072.0,33531072.0
3,cars3d,ed_mixer_t128,38657728.0,38656192.0,4338304.0,4338304.0,38657728.0,34319424.0
4,cars3d,ed_mixer_t256,46138048.0,46136512.0,11031808.0,11031808.0,46138048.0,35106240.0
...,...,...,...,...,...,...,...,...
91,shapes3d,split_resnet,11948594.0,11948594.0,0.0,0.0,11948594.0,11948594.0
92,shapes3d,split_resnet_mixer_alg,11977835.0,11977835.0,0.0,0.0,11977835.0,11977835.0
93,shapes3d,split_resnet_mixer_t128,25765995.0,25765995.0,11031808.0,11031808.0,25765995.0,14734187.0
94,shapes3d,split_resnet_mixer_t256,48984171.0,48984171.0,31496704.0,31496704.0,48984171.0,17487467.0


In [39]:
# Exportar tablas LaTeX separadas por dataset para que no queden demasiado largas

model_order = [
    "resnet18",
    #"resnet18_mixer",
    "resnet18_mixer_alg",
    "resnet18_mixer_t64",
    "resnet18_mixer_t128",
    "resnet18_mixer_t256",
    "split_resnet",
    #"split_resnet_mixer",
    "split_resnet_mixer_alg",
    "split_resnet_mixer_t64",
    "split_resnet_mixer_t128",
    "split_resnet_mixer_t256",
    "split_resnet_mixer_reduced_rep_rep128",
    "split_resnet_mixer_reduced_rep_rep256",
    "ed",
   # "ed_mixer",
    "ed_mixer_t64",
    "ed_mixer_t128",
    "ed_mixer_t256",
    "ed_mixer_alg",
]

df = final_df.copy()
df = df[
    df["dataset"].notna()
    & (df["dataset"] != "-")
    & df["modelo"].isin(model_order)
].copy()

MODEL_NAME_MAP = {
    "resnet18": "ResNet-18",
  #  "resnet18_mixer": "ResNet18 + Mixer",
    "resnet18_mixer_alg": "ResNet-18 + LATTICE(ANG)",
    "resnet18_mixer_t64": "ResNet-18 + LATTICE(TF-64)",
    "resnet18_mixer_t128": "ResNet-18 + LATTICE(TF-128)",
    "resnet18_mixer_t256": "ResNet-18 + LATTICE(TF-256)",
    "split_resnet": "AIN",
   # "split_resnet_mixer": "AIN + Mixer",
    "split_resnet_mixer_alg": "AIN + LATTICE(ANG)",
    "split_resnet_mixer_t64": "AIN + LATTICE(TF-64)",
    "split_resnet_mixer_t128": "AIN + LATTICE(TF-128)",
    "split_resnet_mixer_t256": "AIN + LATTICE(TF-256)",
   # "split_resnet_mixer_reduced_rep_rep128": "AIN + LATTICE(TF-128)",
   # "split_resnet_mixer_reduced_rep_rep256": "AIN + LATTICE(TF-256)",
    "ed": "ED",
    "ed_mixer": "ED + Mixer",
    "ed_mixer_t64": "ED + LATTICE(TF-64)",
    "ed_mixer_t128": "ED + LATTICE(TF-128)",
    "ed_mixer_t256": "ED + LATTICE(TF-256)",
    "ed_mixer_alg": "ED + LATTICE(ANG)"
}

DATASET_NAME_MAP = {
    "cars3d": "Cars3D",
    "dsprites": "dSprites",
    "shapes3d": "Shapes3D",
    "mpi3d": "MPI3D",
    "iraven": "I-RAVEN",
    "clevr": "CLEVR",
}

baseline = (
    df[df["modelo"] == "resnet18"][["dataset", "params_training", "params_inferencia_sin_mixer"]]
    .drop_duplicates("dataset")
    .rename(
        columns={
            "params_training": "resnet18_training",
            "params_inferencia_sin_mixer": "resnet18_inferencia",
        }
    )
)

tab = df.merge(baseline, on="dataset", how="left")
tab["pct_train_vs_resnet18"] = 100.0 * tab["params_training"] / tab["resnet18_training"]
tab["pct_inf_vs_resnet18"] = 100.0 * tab["params_inferencia_sin_mixer"] / tab["resnet18_inferencia"]

tab = tab.assign(
    dataset=tab["dataset"].map(DATASET_NAME_MAP).fillna(tab["dataset"]),
    modelo=tab["modelo"].map(MODEL_NAME_MAP).fillna(tab["modelo"]),
)

model_display_order = [MODEL_NAME_MAP.get(m, m) for m in model_order]
tab["modelo"] = pd.Categorical(tab["modelo"], categories=model_display_order, ordered=True)
tab = tab.sort_values(["dataset", "modelo"]).reset_index(drop=True)


def build_latex_table_for_dataset(dataset_name: str, dataset_tab: pd.DataFrame) -> str:
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{lcc}",
        r"\toprule",
        r"Modelo & \% de parámetros en training & \% de parámetros en inferencia \\",
        r"\midrule",
    ]

    for row in dataset_tab.itertuples(index=False):
        lines.append(
            f"{row.modelo} & {row.pct_train_vs_resnet18:.2f}\% & {row.pct_inf_vs_resnet18:.2f}\% \\\\")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            f"\caption{{{dataset_name}: Training and Inference parameters for multiple models using Resnet-18 as a baseline. LATTICE(TF) imposes heavy parameter increases during training however, during inference its parameter increase is barely superior to AIN. LATTICE(ANG) does not increase computation neither during training nor inference.}}",
            f"\label{{tab:parametros_vs_resnet18_{dataset_name.lower()}}}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

latex_tables_by_dataset = {
    dataset_name: build_latex_table_for_dataset(dataset_name, g)
    for dataset_name, g in tab.groupby("dataset", sort=True)
}

for dataset_name, latex_table in latex_tables_by_dataset.items():
    print(f"\n--- Tabla para {dataset_name} ---\n")
    print(latex_table)


--- Tabla para CLEVR ---

\begin{table}[ht]
\centering
\small
\begin{tabular}{lcc}
\toprule
Modelo & \% de parámetros en training & \% de parámetros en inferencia \\
\midrule
ResNet-18 & 100.00\% & 100.00\% \\
ResNet-18 + LICoG(ANG) & 100.00\% & 100.00\% \\
ResNet-18 + LICoG(TF-64) & 105.62\% & 100.59\% \\
ResNet-18 + LICoG(TF-128) & 111.79\% & 101.18\% \\
ResNet-18 + LICoG(TF-256) & 125.88\% & 102.35\% \\
AIN & 105.25\% & 105.25\% \\
AIN + LICoG(ANG) & 105.33\% & 105.33\% \\
AIN + LICoG(TF-64) & 134.73\% & 111.21\% \\
AIN + LICoG(TF-128) & 173.46\% & 117.07\% \\
AIN + LICoG(TF-256) & 279.04\% & 128.79\% \\
ED & 399.72\% & 399.72\% \\
ED + LICoG(TF-64) & 429.13\% & 405.60\% \\
ED + LICoG(TF-128) & 467.86\% & 411.47\% \\
ED + LICoG(TF-256) & 573.44\% & 423.19\% \\
ED + LICoG(ANG) & 399.72\% & 399.72\% \\
\bottomrule
\end{tabular}
\caption{CLEVR: Training and Inference parameters for multiple models using Resnet-18 as a baseline. LICoG(TF) imposes heavy parameter increases during traini